# Fine-tune Qwen3.5-4B on RBI Regulatory QA Dataset (QLoRA) — Local GPU Edition

**Dataset**: `Magneto/qa_with_personas` — 23,892 Q&A pairs generated from RBI (Reserve Bank of India) circulars and master directions. Despite the dataset card saying "BI", this is RBI data (find-and-replace typo in the original README).

**Format**: RAG-style — `question + context -> answer`. The model learns to answer using a provided regulatory passage, which is the right setup since you'll eventually pair this with a retriever (or just always supply the relevant context at inference time).

**Target model**: `Qwen/Qwen3.5-4B`, QLoRA (4-bit NF4 + LoRA adapters).

**This version runs on your local GPU instead of Google Colab.** The only things that actually change for "local" are: no Colab mounts/magics, no `getpass` prompts assuming a notebook-in-browser UX (still works fine, just noting it), GPU detection generalized beyond T4 (with a VRAM-based micro-batch heuristic for cards like a 6GB RTX 3050), local paths instead of `/content/...`, and a plain local `zip`/copy step at the end instead of a Colab-specific one. All of the modeling/architecture reasoning from the original notebook is preserved as-is, since none of that is Colab-specific.

## This is NOT just a model-name swap from the Qwen3-4B notebook

Qwen3.5 is a genuinely different, newer architecture from Qwen3 — not a version bump of the same
design. The differences that actually affect this notebook's code:

- **Hybrid attention, not pure attention.** Qwen3.5-4B uses a 3:1 stack of Gated DeltaNet
  (linear attention) layers to Gated Attention (full attention) layers — 24 DeltaNet layers,
  8 full-attention layers, 32 total. Only the 8 full-attention layers use the familiar
  `q_proj`/`k_proj`/`v_proj`/`o_proj` naming that LoRA targets by name; the DeltaNet layers have
  a different internal structure. This notebook targets the named attention projections only
  (same approach as the Qwen3 notebook, and the one with the most precedent in published
  fine-tuning work on this exact model) — see the LoRA config cell for the `all-linear`
  alternative and its tradeoffs.
- **Natively multimodal checkpoint, text-only usage.** The `Qwen/Qwen3.5-4B` checkpoint on the
  Hub includes vision-tower weights (it's an image+video+text model). We load it with
  `AutoModelForCausalLM`, which resolves to the text-only `Qwen3_5ForCausalLM` class and skips
  the vision weights entirely — we never touch images here, this is a text-only RAG QA task.
- **Larger vocabulary.** ~248,320 tokens vs Qwen3's ~151,936 — about 1.6x bigger embedding
  table and LM head, with downstream effects on memory and on `MAX_SEQ_LENGTH` choice.

## Local prerequisites (read this before running)

1. **NVIDIA GPU + drivers**: a working NVIDIA driver and CUDA-capable PyTorch install. Run `nvidia-smi` in a terminal first — if that doesn't show your GPU, fix that before touching this notebook.
2. **Python environment**: a dedicated virtualenv or conda env is strongly recommended (this notebook installs/pins specific package versions that can clash with other projects). Activate it, then point your Jupyter kernel at it.
3. **OS note**: `bitsandbytes` 4-bit support on native Windows has historically lagged Linux. If you're on Windows and hit `bitsandbytes` import or CUDA errors, run this under **WSL2** rather than native Windows Python — it'll save you debugging time.
4. **Disk space**: the base model download is ~9 GB; budget a few GB more for checkpoints.
5. This notebook assumes you're running it as a **local Jupyter server** (`jupyter lab` / `jupyter notebook`) or inside VS Code / JupyterLab Desktop — anything that gives you a normal Python kernel with GPU access. There is no Colab runtime involved anywhere below.

## 1. Install dependencies

`Qwen3_5ForCausalLM` is in mainline `transformers` — support landed in **v5.2.0** (confirmed by
the model card and multiple downstream framework changelogs), so this is a real PyPI release
pin, NOT a from-source/bleeding-edge-main install. If your environment somehow resolves an
older `transformers` despite the pin below, the version-check cell right after this one will
fail loudly rather than silently giving you a confusing `model type qwen3_5 not recognized`
error later.

**Local-only note**: run this once in your activated virtualenv/conda env. Unlike Colab, there's
no "runtime" to restart — if pip reports it upgraded `transformers`, just restart your Jupyter
kernel (Kernel → Restart) so the new version is actually imported, then re-run from the top.

In [ ]:
%pip install -q -U "transformers>=5.2.0" accelerate peft bitsandbytes trl datasets evaluate sentencepiece tensorboard matplotlib wandb

In [ ]:
import torch, transformers, peft, trl, bitsandbytes
from packaging import version

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM total: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print("\nWARNING: torch.cuda.is_available() is False. Check `nvidia-smi` and your PyTorch CUDA build "
          "before continuing -- everything below assumes a working local GPU.")

# Qwen3.5 needs transformers >= 5.2.0 (the version that added Qwen3_5ForCausalLM). Failing here
# with a clear message beats failing three cells later at model load with a cryptic
# "model type qwen3_5 not recognized" error.
assert version.parse(transformers.__version__) >= version.parse("5.2.0"), (
    f"transformers {transformers.__version__} is too old for Qwen3.5 (needs >= 5.2.0). "
    f"Re-run the pip install cell above, then restart your Jupyter kernel (Kernel > Restart Kernel), "
    f"then re-run from the top."
)
print("\ntransformers version OK for Qwen3.5.")

### Weights & Biases login

Paste your W&B API key when prompted (find it at https://wandb.ai/authorize). `getpass` hides the
characters as you type and — importantly — does NOT save the key anywhere in this notebook file, so it's
safe to share/upload the `.ipynb` afterward without leaking your key. This works identically in a local
Jupyter kernel as it did in Colab. You only need to do this once per kernel session; it'll stay logged in
until you restart the kernel.

**If you'd rather not use W&B at all**, skip this cell and remove `"wandb"` from `report_to` in the
`SFTConfig` cell later — TensorBoard (local, no account) and the local JSONL logger will still work.

In [ ]:
import wandb
from getpass import getpass

wandb_api_key = getpass("Paste your W&B API key (input hidden): ")
wandb.login(key=wandb_api_key)

# Clear the key from this variable immediately after use -- it's no longer needed,
# and there's no reason to keep a copy of the secret sitting in a live notebook variable.
del wandb_api_key
print("W&B login successful.")

### Detect GPU and set local-appropriate defaults

**Always bf16, regardless of GPU**: Qwen3's native parameters/buffers are bf16, and mixing them
with PyTorch's fp16 `GradScaler` crashes outright (`NotImplementedError:
..._unscale_cuda not implemented for 'BFloat16'`), because the scaler only knows how to handle
fp16 gradients. This isn't a tunable setting for Qwen3 — bf16 training is required regardless of
GPU generation, including consumer cards like a 3050/3060/4060 that lack the fast bf16 tensor
cores of an A100/H100. You'll be slower in bf16 than you would be in fp16 on an fp16-native model,
but that's a speed cost, not a correctness option, here.

**Local-vs-Colab difference**: the original notebook only needed to special-case the Colab T4 (a
fixed, known GPU). Running locally, your card could be anything from a 6GB laptop GPU to a 24GB
desktop card, so the cell below reads your **actual detected VRAM** and picks a micro-batch size
from that, instead of assuming a specific model name. Gradient checkpointing stays on by default
as a memory safety margin across the board (same reasoning as the original notebook: TRL's
per-token entropy logging metric can, on a non-chunked fallback path, materialize a full
`[batch*seq_len, vocab_size]` tensor — ~248k vocab for Qwen3.5 — and checkpointing buys headroom
against that even though the primary chunked-loss path doesn't strictly need it).

In [ ]:
_gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
_vram_gb = (torch.cuda.get_device_properties(0).total_memory / 1e9) if torch.cuda.is_available() else 0.0

# Always bf16: Qwen3's native dtype is bf16, and fp16 + GradScaler is incompatible with it
# (crashes in torch.amp.grad_scaler regardless of GPU generation). Not a tunable choice here.
USE_BF16 = True

# Local heuristic: pick micro-batch size from MEASURED VRAM rather than a hardcoded GPU name list
# (the Colab version only had to handle a handful of known Colab GPU SKUs; locally you could have
# anything). Gradient checkpointing stays on across the board as a safety margin -- see markdown
# above for why this matters even on the chunked-loss path.
USE_GRAD_CHECKPOINTING = True

if _vram_gb == 0:
    SUGGESTED_MICRO_BATCH = 1  # CPU fallback -- training will be extremely slow, but won't crash on this setting
elif _vram_gb < 9:
    # e.g. RTX 3050 6GB/8GB, RTX 2060, GTX 1660 -- tight for a 4B model even at 4-bit + LoRA
    SUGGESTED_MICRO_BATCH = 1
elif _vram_gb < 16:
    # e.g. RTX 3060 12GB, RTX 4060 Ti 16GB, RTX 3080
    SUGGESTED_MICRO_BATCH = 2
else:
    # e.g. RTX 3090/4090 24GB, A100, datacenter cards
    SUGGESTED_MICRO_BATCH = 4

print(f"Detected GPU: {_gpu_name or 'none (CPU only)'}")
print(f"Detected VRAM: {_vram_gb:.1f} GB")
print(f"USE_BF16={USE_BF16}  USE_GRAD_CHECKPOINTING={USE_GRAD_CHECKPOINTING}  "
      f"SUGGESTED_MICRO_BATCH={SUGGESTED_MICRO_BATCH}")
print("These feed into the config cell below -- override there if you know better for your setup.")
if _vram_gb and _vram_gb < 9:
    print("\nNote: under 9GB VRAM is tight for a 4B model. If you OOM even at micro-batch 1, see the "
          "'local troubleshooting' notes near the bottom of this notebook (drop MAX_SEQ_LENGTH, narrow "
          "LORA_TARGET_MODULES, or fall back to Qwen3.5-2B/0.8B if your task allows it).")

### Why these specific numbers? (runtime-budget reasoning, with the uncertainty flagged honestly)

The original Colab version of this notebook targeted a **5.5-hour Colab T4** session ceiling,
because Colab disconnects you after a fixed time. **Running locally, that ceiling doesn't
exist** — your machine is yours for as long as you want to leave it training. So the budget
math below is kept for context (it explains *why* the LoRA/data-volume defaults are sized the
way they are) but the actual constraint locally is just: your GPU, your patience, and your
electricity bill.

**The anchor point**: a real timed run of the sibling Qwen3-4B notebook (attention-only LoRA,
`MAX_SEQ_LENGTH=384`, micro-batch 1, effective batch 16) measured **76.8 seconds/step** on a
Colab T4 — included here purely as a reference point for what "slow" looks like on a similarly
sized GPU; your local card may be faster or slower.

**Why Qwen3.5-4B can't just reuse that number directly:**
- It's a similar total parameter count (~4B) but a genuinely different architecture — a 3:1
  Gated DeltaNet (linear attention) / Gated Attention hybrid, not pure full attention. Linear
  attention layers have a different compute profile than full attention; whether that nets out
  faster or slower per-step than equivalent full-attention layers depends on implementation
  details that vary by hardware.
- ~1.6x larger vocabulary (248,320 vs ~151,936) means a bigger embedding table and LM head,
  which adds memory and compute cost independent of the hyperparameters above.
- The DeltaNet layers have optional fast CUDA kernels (`causal_conv1d`, `fla`) that are **not
  installed by default** here either, unless you install them yourself. If you're doing repeated
  local runs, installing these is worth it — see the local-troubleshooting notes near the bottom.

**What to actually do locally**: run the live-timing probe cell (a few cells before
`trainer.train()`) on your real GPU before committing to a long run. It measures actual
seconds/step on your hardware using the real dataloader/optimizer, so you get a number specific
to your machine instead of a Colab-T4-derived estimate. Use that number, together with
`TRAIN_SUBSET_SIZE` and `NUM_EPOCHS` in the config cell below, to decide how long you're willing
to let it run — and adjust those two knobs (not anything else) up or down once you see your real
per-step timing.

## 2. Config

Set `MODEL_NAME` to try 4B first. If you hit `CUDA out of memory` during training, just change this one line to the 1.7B fallback and re-run from here — no other code changes needed.

**Sized down for a fast smoke test**: `TRAIN_SUBSET_SIZE=200`-scale runs let you confirm the whole pipeline (load, LoRA, train, save, generate) actually completes on your machine before committing to a longer run. Once this finishes cleanly, bump `TRAIN_SUBSET_SIZE` up (or to `None` for the full 19,113-example train set) and re-check `MAX_SEQ_LENGTH` against Cell 3a's real percentiles. Since there's no fixed session-length ceiling locally, feel free to be more generous with `NUM_EPOCHS` and `TRAIN_SUBSET_SIZE` than the Colab defaults below once you've benchmarked your real seconds/step.

In [ ]:
# ============================================================
# MODEL SELECTION
# ============================================================
# This notebook is dedicated to Qwen3.5-4B specifically -- no fallback to a different model.
# The Hub checkpoint is the full multimodal release (it includes vision-tower weights for
# image/video input), but AutoModelForCausalLM resolves "qwen3_5" to the text-only
# Qwen3_5ForCausalLM class, which explicitly skips loading vision weights
# (_keys_to_ignore_on_load_unexpected includes "model.visual.*") -- so we get a clean text-only
# load from the same checkpoint, no separate text-only repo needed.
MODEL_NAME = "Qwen/Qwen3.5-4B"  # ~4 billion parameters (dense, hybrid linear/full attention)

# --- Dataset ---
DATASET_NAME = "Magneto/qa_with_personas"  # the RBI regulatory Q&A dataset on Hugging Face Hub

# ============================================================
# SEQUENCE LENGTH
# ============================================================
# MAX_SEQ_LENGTH = the maximum number of TOKENS the model will see per training example, after
# combining the system prompt + context + question + answer. Qwen3.5 uses a DIFFERENT tokenizer
# than Qwen3 (much larger vocab -- 248,320 vs ~151,936), so don't assume token-length percentiles
# from other notebooks carry over unchanged here. Cell 3a below re-measures the distribution
# using Qwen3.5's own tokenizer -- run it before trusting this number for anything beyond the
# value already set here. A larger vocabulary generally means MORE efficient tokenization (each
# token can represent more text on average), so if anything we'd expect Qwen3.5 to need a
# slightly SHORTER MAX_SEQ_LENGTH than Qwen3 for the same text -- 512 is kept as the starting
# point, but treat Cell 3a's real output as the authority.
MAX_SEQ_LENGTH = 512

# ============================================================
# LoRA CONFIG (the "adapter" -- what actually gets trained)
# ============================================================
# LoRA = Low-Rank Adaptation. Instead of updating all ~4 billion parameters of the base model
# (which would need far more VRAM than most local GPUs have), we freeze the base model entirely
# and insert small trainable "adapter" matrices into specific layers. Only those small matrices
# get updated during training -- typically well under 1% of the total parameter count.

# LORA_R ("rank"): controls the SIZE of each adapter matrix, and therefore how much new
# information the adapter can encode. Rank 8 is a small, common starting point.
LORA_R = 8

# LORA_ALPHA: a scaling factor applied to the adapter's output before it's added back into the
# frozen base model's computation. Conventionally set to 2x the rank (8 -> 16).
LORA_ALPHA = 16

# LORA_DROPOUT: during training, randomly "turns off" this fraction (5%) of the adapter's
# internal connections on each forward pass, as a regularization technique against overfitting.
LORA_DROPOUT = 0.05

# LORA_TARGET_MODULES: WHICH weight matrices inside each transformer layer get an adapter.
# THIS IS THE PART THAT ACTUALLY CHANGES FOR QWEN3.5'S ARCHITECTURE.
#
# Qwen3.5-4B is a 3:1 hybrid: 24 Gated DeltaNet (linear attention) layers for every 8 Gated
# Attention (full attention) layers, 32 total. Only the 8 full-attention layers use the
# familiar q_proj/k_proj/v_proj/o_proj naming -- a published fine-tuning comparison on this
# exact model (S0 Tuning, arXiv 2604.01168) confirms LoRA on Qwen3.5-4B targeting
# {q,k,v,o}_proj hits "the 8 attention layers" specifically, and reports it as a working,
# precedented baseline configuration. The 24 DeltaNet layers have a different internal
# structure (their own query/key/value-like projections plus a short causal convolution) that
# isn't named q_proj/k_proj/v_proj/o_proj, so the named-list approach below simply does not
# touch them.
#
# We use the SAME named-attention-only approach as the Qwen3 notebooks for consistency and
# because it's the better-evidenced choice. The alternative -- target_modules="all-linear" --
# would also adapt the DeltaNet layers' own linear projections (more of the model gets
# adapted, in principle more capacity), but "all-linear" auto-discovery on a hybrid
# architecture this new is less battle-tested, and some Unsloth/community docs note it as
# "optional" rather than required even for full Qwen3.5 fine-tunes. If you want to experiment
# with it, swap the line below -- just know it's the less-precedented path on this specific
# architecture, and budget extra retry time for layer-name surprises.
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
# LORA_TARGET_MODULES = "all-linear"  # alternative: also adapts the 24 DeltaNet layers -- see note above

# ============================================================
# TRAINING CONFIG
# ============================================================
# Local path instead of Colab's /content/... -- this lands in your current working directory.
OUTPUT_DIR = "./qwen3.5-4b-rbi-qa-lora"  # where checkpoints, logs, and the final adapter get saved

# PER_DEVICE_BATCH_SIZE ("micro-batch"): how many training examples are processed together in
# ONE forward+backward pass on the GPU. SUGGESTED_MICRO_BATCH comes from the VRAM-based
# GPU-detection cell above. Note: Qwen3.5-4B's larger vocab/LM-head means somewhat higher
# activation memory than Qwen3-4B at the same micro-batch and MAX_SEQ_LENGTH -- if you OOM even
# at micro-batch 1, the next lever to pull is MAX_SEQ_LENGTH, not this value (it's already at
# the floor).
PER_DEVICE_BATCH_SIZE = SUGGESTED_MICRO_BATCH

# GRAD_ACCUM_STEPS: kept at the same effective-batch target (16) as the other notebooks in this
# series, for easy side-by-side comparison.
GRAD_ACCUM_STEPS = 16 // PER_DEVICE_BATCH_SIZE

# NUM_EPOCHS: 1 by default, matching the original Colab notebook's conservative choice given the
# genuine uncertainty in this architecture's per-step timing. Locally, since there's no fixed
# session-length ceiling, feel free to raise this to 2+ once you've benchmarked your real
# seconds/step with the probe cell below and decided you have time to spare.
NUM_EPOCHS = 1

# LEARNING_RATE: 2e-4 is a standard LoRA learning rate.
LEARNING_RATE = 2e-4

# WARMUP_RATIO: first 3% of steps ramp up linearly.
WARMUP_RATIO = 0.03

# LOGGING_STEPS: how often to print/record a loss value.
LOGGING_STEPS = 10

# SAVE_STEPS / EVAL_STEPS: checkpoint and validation-eval frequency.
SAVE_STEPS = 50
EVAL_STEPS = 50

# ============================================================
# DATA VOLUME
# ============================================================
# TRAIN_SUBSET_SIZE=3000, NUM_EPOCHS=1, effective batch 16 -> ~190 training steps -- this was
# sized against a Colab 5.5h ceiling in the original notebook. Locally there's no such ceiling:
# run the live-timing probe cell below first to see your real seconds/step, then raise
# TRAIN_SUBSET_SIZE (or set it to None for the full 19,113-example train set) based on how long
# you're actually willing to let your machine run.
TRAIN_SUBSET_SIZE = 3000
EVAL_SUBSET_SIZE = 200

# ============================================================
# EXPERIMENT TRACKING (Weights & Biases + TensorBoard)
# ============================================================
WANDB_PROJECT = "qwen3-rbi-qa-finetune"

import time as _time
RUN_NAME = (
    f"{MODEL_NAME.split('/')[-1]}"
    f"_subset{TRAIN_SUBSET_SIZE if TRAIN_SUBSET_SIZE else 'FULL'}"
    f"_ep{NUM_EPOCHS}"
    f"_{_time.strftime('%Y%m%d-%H%M%S')}"
)
print("This run will be logged as:", RUN_NAME)

## 3. Load and inspect the dataset

By default, `load_dataset` caches downloaded parquet files under `~/.cache/huggingface/datasets/`.
On Colab, this lived on the ephemeral VM disk and got wiped on every runtime restart -- which is
why the original notebook offered an optional Google Drive cache. **Running locally, this
problem doesn't exist**: your `~/.cache/huggingface/datasets/` directory is on your own normal
disk and persists across kernel restarts and reboots just like any other file, so the dataset
only ever needs to download once. No Drive mount, no extra auth step -- it's just removed here.

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset(DATASET_NAME)
print(raw_dataset)
print()
print("Example record:")
print(raw_dataset["train"][0])

### 3a. Check token length distribution before fixing MAX_SEQ_LENGTH

This matters because the dataset card only tells us `context` is ~1000 chars — we should verify actual tokenized length (question + context + answer) before committing to a sequence length, otherwise we either waste VRAM on padding or silently truncate answers. **This re-measurement matters more here than in the Qwen3 notebooks**: Qwen3.5 uses a different tokenizer with a ~1.6x larger vocabulary (248,320 vs ~151,936 tokens), so the token counts for the same text will differ from what was measured for Qwen3 -- don't assume the `MAX_SEQ_LENGTH=512` default carries over unchanged without checking this cell's output.

In [ ]:
from transformers import AutoTokenizer
import numpy as np

_tok_probe = AutoTokenizer.from_pretrained(MODEL_NAME)

sample = raw_dataset["train"].shuffle(seed=42).select(range(min(1000, len(raw_dataset["train"]))))
lengths = []
for ex in sample:
    full_text = f"{ex['context']}\n\n{ex['question']}\n\n{ex['answer']}"
    lengths.append(len(_tok_probe(full_text)["input_ids"]))

lengths = np.array(lengths)
print(f"mean={lengths.mean():.0f}  median={np.median(lengths):.0f}  "
      f"p90={np.percentile(lengths, 90):.0f}  p95={np.percentile(lengths, 95):.0f}  "
      f"p99={np.percentile(lengths, 99):.0f}  max={lengths.max()}")
print()
print("If p95 is well under MAX_SEQ_LENGTH, you're safe. If not, either raise "
      "MAX_SEQ_LENGTH (costs VRAM) or accept truncation on the longest tail examples.")

## 4. Format for RAG-style instruction tuning

Each example becomes a chat-formatted sequence: a system prompt establishing the assistant's role, a user turn with the regulatory context + question, and an assistant turn with the answer. We use Qwen3's chat template with `enable_thinking=False` since this is direct QA, not a reasoning task — we don't want the model generating `<think>` blocks for straightforward regulatory lookups.

In [ ]:
SYSTEM_PROMPT = (
    "You are a knowledgeable assistant on Reserve Bank of India (RBI) banking regulations. "
    "Answer the question using only the information in the provided regulatory context. "
    "Be precise and comprehensive, and note when the context does not fully answer the question."
)

def build_messages(example):
    user_content = f"Context:\n{example['context']}\n\nQuestion: {example['question']}"
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": example["answer"]},
        ]
    }

train_ds = raw_dataset["train"].map(build_messages, remove_columns=raw_dataset["train"].column_names)
eval_ds = raw_dataset["validation"].map(build_messages, remove_columns=raw_dataset["validation"].column_names)

if TRAIN_SUBSET_SIZE is not None:
    train_ds = train_ds.shuffle(seed=42).select(range(min(TRAIN_SUBSET_SIZE, len(train_ds))))
if EVAL_SUBSET_SIZE is not None:
    eval_ds = eval_ds.shuffle(seed=42).select(range(min(EVAL_SUBSET_SIZE, len(eval_ds))))

print(f"train: {len(train_ds)}  |  eval: {len(eval_ds)}")
print()
print("Example formatted record:")
print(train_ds[0])

## 5. Load model in 4-bit (QLoRA)

Loads `Qwen/Qwen3.5-4B` directly -- no fallback model here (this notebook is dedicated to this
one model). `AutoModelForCausalLM` resolves the `qwen3_5` model type to the text-only
`Qwen3_5ForCausalLM` class and skips loading the vision-tower weights automatically (verified
in the cell below by checking that no `visual`/`vision` keys ended up in the loaded model and
that the model's class name is the one we expect) -- so even though the checkpoint includes
multimodal weights, what actually lands on your GPU is a text-only decoder.

**Local note**: the `~9 GB` download happens once and is cached under
`~/.cache/huggingface/hub/` on your own disk -- subsequent runs (even after a kernel restart)
reuse it, no re-download needed.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

_compute_dtype = torch.bfloat16 if USE_BF16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=_compute_dtype,
    bnb_4bit_use_double_quant=True,
)

def load_model_and_tokenizer(model_name):
    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=_compute_dtype,
    )
    model.config.use_cache = False  # required for gradient checkpointing; re-enabled before generation later
    return model, tok

# No try/except fallback here -- this notebook is dedicated to Qwen3.5-4B. If you OOM even at
# 4-bit on your local GPU, restart the kernel (to fully release VRAM) and try dropping
# MAX_SEQ_LENGTH in the config cell, or see the local troubleshooting notes near the bottom.
active_model_name = MODEL_NAME
model, tokenizer = load_model_and_tokenizer(MODEL_NAME)
print(f"Loaded {MODEL_NAME} successfully.")
print("Active model:", active_model_name)
print("GPU memory allocated: %.2f GB" % (torch.cuda.memory_allocated() / 1e9))

# --- Defensive check: confirm this actually loaded as text-only, not the full multimodal model ---
# Qwen3_5ForCausalLM should have no vision-tower submodule. If this assertion ever fails, it
# means a transformers version change resolved AutoModelForCausalLM differently than expected --
# better to catch that here than discover it three cells later as a confusing shape mismatch.
_model_class_name = type(model).__name__
_has_vision_weights = any("visual" in name or "vision" in name for name, _ in model.named_parameters())
print(f"\nLoaded model class: {_model_class_name}")
print(f"Vision-tower weights present: {_has_vision_weights} (expected: False)")
assert _model_class_name == "Qwen3_5ForCausalLM", (
    f"Expected Qwen3_5ForCausalLM, got {_model_class_name} -- check your transformers version."
)
assert not _has_vision_weights, "Unexpected vision-tower weights found in a supposedly text-only load."
print("Confirmed: text-only load, no vision-tower weights.")

## 6. Prepare for k-bit training and define the LoRA config

**Important**: we deliberately do NOT call `get_peft_model()` here. `SFTTrainer` applies its memory-efficient
chunked cross-entropy patch (`loss_type="chunked_nll"`, the default) by patching `.forward()` on the inner
causal LM. For that patch to land correctly on a LoRA-wrapped model, TRL needs to do the PEFT-wrapping itself
(via `peft_config=` below) — wrapping the model ourselves first and handing `SFTTrainer` an already-PeftModel
instance can cause the chunked patch to silently miss, which is what produced an earlier
`entropy_from_logits` OOM in development of this notebook: the model fell back to computing full, unchunked
`[batch*seq, vocab]` logits.

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=USE_GRAD_CHECKPOINTING)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,  # do NOT include "lm_head" here -- incompatible with loss_type="chunked_nll"
    bias="none",
    task_type="CAUSAL_LM",
)

print("LoRA config ready (not yet applied). SFTTrainer will wrap the model with this in the next step.")

## 7. Tokenize with chat template (thinking mode OFF)

Qwen3.5 thinks by default (more aggressively than Qwen3 -- it's described as the model's primary mode, not just an option), but the same `enable_thinking=False` argument to `apply_chat_template` disables it, confirmed against Qwen3.5 usage examples. This is RAG QA, not multi-step reasoning, so non-thinking mode is the right fit here, same as Qwen3.

In [ ]:
def formatting_func(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    return text

# Sanity check on one example
print(formatting_func(train_ds[0])[:1500])

## 8. Train with TRL's SFTTrainer

`paged_adamw_8bit` is the key setting that makes this fit on small VRAM — it streams optimizer states to CPU when GPU memory is tight, at a small speed cost. Don't swap this for plain `adamw_torch` on a 6GB card.

**Note**: newer TRL versions renamed `SFTConfig`'s `max_seq_length` argument to `max_length` (the old name was deprecated, then removed). This notebook uses `max_length`. If you're on an older pinned TRL (<0.20) and get an unrecognized-argument error in the opposite direction, swap it back to `max_seq_length`. Run `import trl; print(trl.__version__)` if you want to check which one applies.

In [ ]:
from trl import SFTTrainer, SFTConfig
import os

# WANDB_PROJECT is read automatically by transformers/accelerate from this environment variable --
# it's how the project name set in the config cell actually reaches W&B's servers.
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,                              # checkpoints/logs saved here

    # --- Batch sizing (see config cell for full explanation of micro-batch vs effective batch) ---
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    # --- Core training hyperparameters ---
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    # cosine schedule: the learning rate follows a smooth downward curve (shaped like one hump
    # of a cosine wave) from its peak down to ~0 by the end of training, rather than staying flat
    # the whole time. This tends to produce slightly better final results than a constant rate,
    # since the model takes smaller, more careful steps as it gets closer to convergence.
    lr_scheduler_type="cosine",

    # --- How often to log / checkpoint / evaluate (see config cell for exact meanings) ---
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    eval_strategy="steps",     # evaluate every EVAL_STEPS steps (not just once per epoch)
    save_strategy="steps",     # checkpoint every SAVE_STEPS steps
    save_total_limit=2,        # only keep the 2 most recent checkpoints on disk (saves space)

    # --- Memory-saving settings (important for fitting on limited VRAM) ---
    # paged_adamw_8bit: a memory-efficient version of the Adam optimizer. Normal Adam keeps two
    # extra numbers (momentum + variance) PER PARAMETER, which doubles memory usage on top of the
    # parameters themselves. This variant stores those extra numbers in 8-bit instead of 32-bit,
    # AND can temporarily move them to CPU RAM ("paging") if GPU memory gets tight -- similar to
    # how an OS pages memory to disk under pressure. This is the single most important setting
    # for fitting training on a small GPU.
    optim="paged_adamw_8bit",

    # bf16/fp16: which reduced-precision number format to train in (instead of full 32-bit floats,
    # which would use 2x the memory). USE_BF16 is decided automatically earlier in the notebook --
    # currently always True, because Qwen3's own weights are natively bf16 and training in fp16
    # instead causes a crash (see the GPU-detection cell's comments for the full explanation).
    bf16=USE_BF16,
    fp16=not USE_BF16,

    # gradient_checkpointing: normally, during the forward pass, the model keeps every
    # intermediate calculation in memory so it can use them later during backpropagation. Instead,
    # this setting throws most of them away and RECOMPUTES them when needed -- trading extra
    # computation time for a large reduction in memory usage. USE_GRAD_CHECKPOINTING is set by the
    # GPU-detection cell (currently always on, as a safety margin).
    gradient_checkpointing=USE_GRAD_CHECKPOINTING,
    gradient_checkpointing_kwargs={"use_reentrant": False} if USE_GRAD_CHECKPOINTING else None,

    # max_length: the sequence-length cap explained in detail in the config cell above.
    max_length=MAX_SEQ_LENGTH,

    # assistant_only_loss: ensures the model is only "graded" (loss computed) on the tokens it
    # itself needs to generate -- the answer -- and NOT on the system prompt or the question.
    #
    # IMPORTANT FOR QWEN3.5: this only works correctly because we explicitly pass
    # processing_class=tokenizer to SFTTrainer below. Without that, SFTTrainer calls
    # AutoProcessor.from_pretrained(model_id) to auto-detect a processing class -- and because
    # the Qwen/Qwen3.5-4B Hub repo is the full multimodal checkpoint, AutoProcessor resolves to
    # a multimodal ProcessorMixin REGARDLESS of which model class we actually instantiated
    # (Qwen3_5ForCausalLM, text-only). SFTTrainer treats getting back a ProcessorMixin as proof
    # the model is a VLM (self._is_vlm = True) and behaves accordingly from then on: it blocks
    # assistant_only_loss=True outright, and separately, its chunked-loss patch reads
    # model.config.text_config (the nested-config layout multimodal wrappers use) instead of
    # model.config directly -- which crashes with "'Qwen3_5TextConfig' object has no attribute
    # 'text_config'" because our model's config IS the text config already, not a wrapper around
    # one. Passing processing_class=tokenizer explicitly skips the AutoProcessor auto-detection,
    # so SFTTrainer correctly sets self._is_vlm = False and both problems go away.
    assistant_only_loss=True,

    # --- Experiment tracking: send live metrics to BOTH TensorBoard (local, no account needed)
    # AND Weights & Biases (cloud dashboard, needs the login from the cell near the top). Having
    # both means you can watch live in-notebook (TensorBoard) AND keep a permanent, shareable,
    # comparison-friendly record across runs (W&B) without extra effort.
    report_to=["tensorboard", "wandb"],
    logging_dir=f"{OUTPUT_DIR}/tb_logs",   # where TensorBoard's local log files go
    logging_strategy="steps",
    run_name=RUN_NAME,                      # the human-readable run label set in the config cell

    # load_best_model_at_end + metric_for_best_model: at the very end of training, instead of
    # keeping whichever checkpoint happened to be saved last, reload whichever checkpoint had the
    # LOWEST validation loss seen at any point during training. This protects against the case
    # where the model started overfitting near the end and its final state is actually worse than
    # an earlier one.
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # loss_type="chunked_nll": computes the cross-entropy loss in small chunks instead of all at
    # once. Qwen3.5's vocabulary is ~248,000 possible tokens (about 1.6x Qwen3's ~152,000) --
    # computing the loss the "normal" way would require building a
    # [batch_size x sequence_length, 248000] tensor in one shot, which is even larger here than
    # for the Qwen3 notebooks and was the direct cause of an out-of-memory crash in an earlier
    # version of the Qwen3 notebook this one is based on. Chunking keeps memory usage flat
    # regardless of sequence length or vocab size.
    loss_type="chunked_nll",
)

import json
from transformers import TrainerCallback

class JSONLLoggerCallback(TrainerCallback):
    '''A third logging destination, alongside TensorBoard and W&B: a plain local text file.

    Every time the trainer logs a metric (loss, eval_loss, learning_rate, etc. -- the same
    numbers going to TensorBoard/W&B), this callback also appends it as one line of JSON to
    `training_log.jsonl`. This is useful because it's a plain file you can open, grep, or load
    with pandas at any time, completely independent of whether TensorBoard or W&B are working --
    a good fallback if either of those has a hiccup. Locally this file just lives on your own
    disk in OUTPUT_DIR alongside the checkpoints, so there's no "VM gets wiped" concern the way
    there was on Colab -- it persists exactly as long as you keep the directory around.
    '''
    def __init__(self, log_path):
        self.log_path = log_path
        open(self.log_path, "w").close()  # create/empty the file fresh at the start of this run

    def on_log(self, args, state, control, logs=None, **kwargs):
        # `on_log` is automatically called by the Trainer every time it has new metrics to report
        # (i.e. every LOGGING_STEPS steps, and at each evaluation). `logs` is a dict like
        # {"loss": 1.83, "learning_rate": 0.00019, ...} or {"eval_loss": 1.71} during evaluation.
        if logs is None:
            return
        record = dict(logs)
        record["step"] = state.global_step   # which training step this metric was recorded at
        record["epoch"] = state.epoch         # how many full passes over the data, as a fraction
        with open(self.log_path, "a") as f:
            f.write(json.dumps(record) + "\n")

TRAINING_LOG_PATH = f"{OUTPUT_DIR}/training_log.jsonl"
os.makedirs(OUTPUT_DIR, exist_ok=True)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    formatting_func=formatting_func,
    # processing_class=tokenizer: explicitly pass our already-loaded tokenizer instead of
    # letting SFTTrainer auto-detect a processing class via AutoProcessor.from_pretrained(). On
    # Qwen3.5 specifically, that auto-detection resolves to a multimodal processor (because the
    # Hub repo is the full multimodal checkpoint) even though we loaded the text-only model
    # class -- which makes SFTTrainer misclassify this as a VLM and breaks both
    # assistant_only_loss and the chunked-loss patch. See the comment on assistant_only_loss
    # above for the full explanation. Passing the tokenizer directly sidesteps this entirely.
    processing_class=tokenizer,
    # peft_config (not a manually pre-wrapped model!): SFTTrainer applies the LoRA wrapping
    # itself when given the config this way, which is required for its internal memory-saving
    # patches (like chunked_nll above) to correctly recognize the model's structure. Wrapping the
    # model ourselves first caused a silent failure earlier in this notebook's development.
    peft_config=lora_config,
    callbacks=[JSONLLoggerCallback(TRAINING_LOG_PATH)],
)

# Sanity check: confirms LoRA is actually active and shows what fraction of the model's
# parameters are trainable. Expect well under 1% (typically 0.1-0.5%) -- if this prints something
# close to 100%, LoRA did not get applied correctly and you'd effectively be full-fine-tuning,
# which would be far too slow/memory-hungry for this setup.
trainer.model.print_trainable_parameters()
print(f"W&B project: {WANDB_PROJECT}  |  Run name: {RUN_NAME}")
print(f"Training metrics will also be appended locally to: {TRAINING_LOG_PATH}")

### (Optional) Launch TensorBoard inline

Run this before or during training to watch loss curves live inside the notebook. It reads from
`logging_dir` (`{OUTPUT_DIR}/tb_logs`), which `SFTConfig`'s `report_to="tensorboard"` writes to automatically
— no extra account or API key needed, unlike Weights & Biases. This magic command works the same way in a
local Jupyter session as it did on Colab.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}/tb_logs

### Diagnostic: confirm which precision mode is actually active

Run this once before `trainer.train()`. It checks every layer where an fp16 vs bf16 mismatch could be
hiding: the `ACCELERATE_MIXED_PRECISION` env var (which `TrainingArguments`/`SFTConfig` sets as a process-wide
side effect and which can persist oddly across runs in the same kernel), the resolved `accelerate` state, our
own config flags, and — most importantly — the actual dtype of every distinct parameter/buffer group in the
live model, since PEFT or `bitsandbytes` could be creating something in fp16 regardless of what we asked for.

In [ ]:
import os
from collections import Counter

print("--- Env var ---")
print("ACCELERATE_MIXED_PRECISION =", os.environ.get("ACCELERATE_MIXED_PRECISION", "<not set>"))

print("\n--- Our config flags ---")
print("USE_BF16 =", USE_BF16)
print("sft_config.bf16 =", sft_config.bf16, " sft_config.fp16 =", sft_config.fp16)

print("\n--- Live accelerate state (the actual source of truth for GradScaler) ---")
try:
    accel_state = trainer.accelerator.state
    print("accelerator.mixed_precision =", trainer.accelerator.mixed_precision)
    print("accelerator.scaler =", trainer.accelerator.scaler)
except Exception as e:
    print("Could not read trainer.accelerator state:", e)

print("\n--- Actual parameter/buffer dtypes in the live model (grouped by dtype) ---")
dtype_counts = Counter()
dtype_examples = {}
for name, p in trainer.model.named_parameters():
    dtype_counts[p.dtype] += 1
    if p.dtype not in dtype_examples:
        dtype_examples[p.dtype] = name
for dtype, count in dtype_counts.items():
    print(f"{dtype}: {count} params  (example: {dtype_examples[dtype]})")

print("\nIf you see torch.float16 anywhere above alongside bf16, or if "
      "ACCELERATE_MIXED_PRECISION says 'fp16', that's worth digging into before training.")

### Live runtime check on YOUR hardware (read this one)

Runs a handful of real forward+backward steps through `trainer`'s own dataloader and optimizer
-- the same code path `trainer.train()` will use -- to measure actual seconds/step on *your*
GPU, without permanently consuming any training steps or touching the LR scheduler/W&B/TensorBoard
logging. This replaces the original notebook's "compare against a fixed Colab budget" framing:
locally, there's no session-length ceiling, so the point of this cell is simply to give you a
real number so you can decide how big a run you actually want, by adjusting `TRAIN_SUBSET_SIZE`
and `NUM_EPOCHS` in the config cell above and re-running from there if your projected time is
more than you want to commit to right now.

In [ ]:
import time, math, torch

_PROBE_STEPS = 5
_probe_loader = trainer.get_train_dataloader()
_probe_iter = iter(_probe_loader)

trainer.model.train()
_t0 = time.time()
for _ in range(_PROBE_STEPS):
    _batch = next(_probe_iter)
    _batch = {k: v.to(trainer.model.device) if hasattr(v, "to") else v for k, v in _batch.items()}
    _loss = trainer.compute_loss(trainer.model, _batch)
    if isinstance(_loss, tuple):
        _loss = _loss[0]
        _loss.backward()
    trainer.model.zero_grad(set_to_none=True)  # discard probe gradients -- this run doesn't count
torch.cuda.synchronize()
_elapsed = time.time() - _t0
_measured_sec_per_step = _elapsed / _PROBE_STEPS

_steps_per_epoch = math.ceil(len(train_ds) / (PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS))
_total_steps = _steps_per_epoch * NUM_EPOCHS
_projected_total_hours = (_measured_sec_per_step * _total_steps) / 3600

print(f"Measured: {_measured_sec_per_step:.1f} sec/step over a {_PROBE_STEPS}-step probe on this GPU")
print(f"Planned total steps for the real run: {_total_steps}")
print(f"Projected training time at this speed: {_projected_total_hours:.2f}h")
print()
if _projected_total_hours > 6:
    print("That's a fairly long run. If you'd rather not commit to it right now, lower "
          "TRAIN_SUBSET_SIZE in the config cell and re-run from there -- there's no fixed deadline "
          "locally, this is purely about your own time/electricity tradeoff.")
else:
    print("Looks like a reasonable length of time to let this run locally.")

torch.cuda.empty_cache()

In [ ]:
import torch
torch.cuda.empty_cache()
print("Starting the full training run (the probe above doesn't count toward this -- gradients "
      "from the probe were discarded and the LR scheduler/optimizer haven't taken any real steps).")
print("This will run until completion -- feel free to leave it and check back, or watch live via "
      "the TensorBoard cell above or your W&B dashboard.")
trainer.train()

## 9. Save the LoRA adapter

In [ ]:
FINAL_ADAPTER_DIR = OUTPUT_DIR + "-final"
trainer.save_model(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print("Saved adapter + tokenizer to", FINAL_ADAPTER_DIR)
print("Trained on:", active_model_name)

# Explicitly close out the W&B run now that training + saving are done. Without this, the run
# can be left showing as "still running" on your wandb.ai dashboard until it eventually times
# out on its own -- calling finish() marks it complete immediately and uploads any remaining
# buffered data.
import wandb
wandb.finish()

## 10. Quick qualitative test

In [ ]:
# Use trainer.model -- the actual LoRA-wrapped, trained model. The bare `model` variable
# from the model-loading cell is the unwrapped base model and was never trained (SFTTrainer
# applied the LoRA wrapping internally via peft_config=).
gen_model = trainer.model
gen_model.eval()
gen_model.config.use_cache = True  # was disabled for training (gradient checkpointing); re-enable for fast generation

test_example = eval_ds[0]["messages"][:2]  # system + user only, drop the gold answer
prompt_text = tokenizer.apply_chat_template(
    test_example, tokenize=False, add_generation_prompt=True, enable_thinking=False
)
inputs = tokenizer(prompt_text, return_tensors="pt").to(gen_model.device)

with torch.no_grad():
    output_ids = gen_model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
        temperature=None,
        top_p=None,
    )

generated = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("QUESTION CONTEXT:\n", test_example[1]["content"][:500], "...\n")
print("MODEL ANSWER:\n", generated, "\n")
print("GOLD ANSWER:\n", eval_ds[0]["messages"][2]["content"])

## 11. Review training logs

Reads back `training_log.jsonl` (written live by `JSONLLoggerCallback` during training) and plots train/eval
loss. This works even if the TensorBoard cell above was never opened, and gives you a plain file on your own
disk you can keep alongside the model, independent of whether TensorBoard or W&B are running.

In [ ]:
import json
import matplotlib.pyplot as plt

records = []
with open(TRAINING_LOG_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Loaded {len(records)} logged events from {TRAINING_LOG_PATH}")
print()

train_steps = [r["step"] for r in records if "loss" in r]
train_loss = [r["loss"] for r in records if "loss" in r]
eval_steps = [r["step"] for r in records if "eval_loss" in r]
eval_loss = [r["eval_loss"] for r in records if "eval_loss" in r]

if train_loss:
    plt.figure(figsize=(8, 4))
    plt.plot(train_steps, train_loss, label="train loss", marker="o", markersize=3)
    if eval_loss:
        plt.plot(eval_steps, eval_loss, label="eval loss", marker="s", markersize=4)
    plt.xlabel("step")
    plt.ylabel("loss")
    plt.title(f"Training run: {active_model_name}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("No training-loss events found yet -- run trainer.train() first, or check "
          "LOGGING_STEPS isn't larger than your total step count.")

## Local troubleshooting notes

You're already running locally, so most of these are just "if something goes wrong, try this"
rather than a migration checklist:

1. **OOM at micro-batch 1**: Qwen3.5-4B's larger vocabulary (~248K vs Qwen3's ~152K) means a
   somewhat bigger embedding table and LM head than Qwen3-4B at the same hyperparameters -- if
   you OOM where a same-sized Qwen3 model didn't, that extra vocab overhead is the most likely
   first culprit. Try, in order: (a) restart the kernel to fully clear VRAM, then drop
   `MAX_SEQ_LENGTH` to 256 in the config cell, (b) narrow `LORA_TARGET_MODULES` to just
   `["q_proj", "v_proj"]`, (c) switch to `Qwen/Qwen3.5-2B` or `Qwen/Qwen3.5-0.8B` if available for
   your use case.
2. **Close other GPU-using applications** (browser hardware acceleration, games, other
   notebooks/models) before running -- on a 6-8GB card, 500MB of background usage is the
   difference between fitting and OOMing.
3. **Windows + bitsandbytes**: 4-bit support on native Windows has historically lagged Linux. If
   you hit `bitsandbytes` import or CUDA errors, run this under WSL2 rather than native Windows
   Python.
4. **Resuming after an interruption**: this notebook checkpoints every `SAVE_STEPS` steps -- if
   your local training session gets interrupted (power loss, accidental kernel restart, etc.),
   you can resume with `trainer.train(resume_from_checkpoint=True)`.
5. **Optional speed-up for repeated local runs**: if you install the `causal_conv1d` and `fla`
   packages (the optional fast CUDA kernels for the Gated DeltaNet layers -- not installed by
   default in the pip cell at the top), Qwen3.5-4B may train noticeably faster on your hardware.
   Worth doing if you're iterating on this notebook repeatedly rather than running it once.
6. **Multi-GPU machines**: this notebook's `device_map="auto"` will happily spread the model
   across multiple GPUs if you have them, but the per-step timing probe and VRAM-based
   micro-batch heuristic above only look at `cuda:0`. If you have multiple GPUs and want to pin
   to a specific one, set `CUDA_VISIBLE_DEVICES` in your shell before launching Jupyter, or set
   it via `os.environ["CUDA_VISIBLE_DEVICES"] = "0"` as the very first cell in this notebook
   (before importing torch).

In [ ]:
import shutil

# Local equivalent of the original notebook's Colab zip step -- this just bundles the saved
# adapter directory into a single zip file in your current working directory, handy for backing
# up or moving the adapter to another machine. Skip this cell entirely if you don't need a zip;
# FINAL_ADAPTER_DIR (set above when you saved the adapter) is already a normal local folder you
# can copy, move, or load from directly.
_zip_base_name = "qwen3.5-4b-rbi-qa-lora-final"  # produces qwen3.5-4b-rbi-qa-lora-final.zip
shutil.make_archive(_zip_base_name, "zip", FINAL_ADAPTER_DIR)
print(f"Zipped {FINAL_ADAPTER_DIR} -> {_zip_base_name}.zip")